# TD 6 — Élan et surapprentissage : lire les courbes d'apprentissage

## Objectifs

À la fin de la séance, vous saurez :

- distinguer l'**erreur d'optimisation** (coût d'entraînement) de l'**écart train/test** (capacité à généraliser) ;
- reconnaître les **régimes** d'une courbe d'apprentissage : descente, plateau, remontée (surapprentissage), divergence ;
- **prédire puis mesurer** l'effet du coefficient d'élan $\mu$ sur la dynamique d'apprentissage ;
- distinguer un **zigzag systématique** (causé par l'élan) d'un **bruit de batch** (causé par les mini-lots).

## Plan de séance (≈ 3 h)

1. **Ex. 1** — Premier contact : comparaison des courbes train/test avec et sans élan sur le jeu 2D standard.
2. **Ex. 2** — La vallée mal conditionnée : zigzag, accélération ×10, divergence à grand pas.
3. **Ex. 3** — Le bruit mémorisé : élan et surapprentissage.
4. **Ex. 4** (optionnel) — Mini-batchs : l'élan lisse-t-il vraiment ?
5. **Bilan** — synthèse et règles de lecture des courbes.

## Rappel : règle de mise à jour avec élan (momentum)

$$v_l \leftarrow \mu\, v_l + \eta\, \frac{\partial E}{\partial W_l}, \qquad W_l \leftarrow W_l - v_l$$

La **vitesse** $v_l$ accumule les gradients passés, pondérés par le facteur d'élan $\mu \in [0, 1[$ : le pas effectif est plus grand quand le gradient garde le même sens, et les oscillations sont amorties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy

# Alias de types pour les tableaux numpy
Vec = np.ndarray[tuple[int], np.dtype[np.float64]]        # vecteur 1D de flottants
Mat = np.ndarray[tuple[int, int], np.dtype[np.float64]]   # matrice 2D de flottants

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_prime(z: Mat) -> Mat:
    """
    σ'(z) = σ(z) · (1 - σ(z)).
    """
    s = sigmoid(z)
    return s * (1.0 - s)

def tanh_prime(z: Mat) -> Mat:
    """
    tanh'(z) = 1 - tanh²(z).
    """
    return 1.0 - np.tanh(z) ** 2

ACTIVATIONS = {
    "sigmoid": (sigmoid, sigmoid_prime),
    "tanh": (np.tanh, tanh_prime),
    "linear": (lambda z: z, lambda z: np.ones_like(z)),
}

In [ ]:
class MLP:
    def __init__(self, sizes, activation="sigmoid", output_activation="linear", seed=0):
        rng = np.random.default_rng(seed)
        self.sizes = list(sizes)
        self.Ws = []
        self.bs = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.Ws.append(rng.normal(0.0, 0.5, size=(n_out, n_in)))
            self.bs.append(np.zeros(n_out))
        self.activation, self.activation_prime = ACTIVATIONS[activation]
        self.output_activation, self.output_activation_prime = ACTIVATIONS[output_activation]

    def predict(self, X):
        return self.forward(X)[0][-1]

    def loss(self, X: Mat, t: Mat) -> float:
        """Coût du modèle : erreur quadratique moyenne.

        E = (1/2) · mean(Σ_k (Y_k - T_k)²).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        float
            Coût quadratique moyen sur le batch.
        """
        # coût du cours : moyenne sur les échantillons de 1/2 · Σ_k (Y_k - T_k)^2
        return 0.5 * float(np.mean(np.sum((self.predict(X) - t) ** 2, axis=1)))

    def forward(self, X: Mat) -> tuple[list[Mat], list[Mat]]:
        """Passe avant du MLP, couche par couche.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).

        Returns
        -------
        tuple[list[Mat], list[Mat]]
            (As, Zs) : activations (avec Y0 = X en tête) et pré-activations
            de chaque couche.
        """
        A = X
        As = [A]
        Zs = []
        n = len(self.Ws)
        for l, (W, b) in enumerate(zip(self.Ws, self.bs)):
            Z = A @ W.T + b
            Zs.append(Z)
            f = self.output_activation if l == n - 1 else self.activation
            A = f(Z)
            As.append(A)
        return As, Zs

    def backward(self, X: Mat, t: Mat) -> list[tuple[Mat, Vec]]:
        """Passe arrière : gradients du coût pour chaque couche.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        list[tuple[Mat, Vec]]
            Liste de (dW_l, db_l), une paire de gradients par couche.
        """
        As, Zs = self.forward(X)
        N = X.shape[0]
        n = len(self.Ws)
        delta = (As[-1] - t) * self.output_activation_prime(Zs[-1])
        dWs, dbs = [], []
        for l in range(n - 1, -1, -1):
            dW = delta.T @ As[l] / N
            db = delta.sum(axis=0) / N
            dWs.insert(0, dW)
            dbs.insert(0, db)
            if l > 0:
                delta = (delta @ self.Ws[l]) * self.activation_prime(Zs[l - 1])
        return list(zip(dWs, dbs))

    def step(self, X: Mat, t: Mat, lr: float) -> None:
        """Effectue une étape de descente de gradient (mise à jour des poids).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).
        lr : float
            Pas d'apprentissage.
        """
        for l, (dW, db) in enumerate(self.backward(X, t)):
            self.Ws[l] -= lr * dW
            self.bs[l] -= lr * db

    def fit_batch(self, X: Mat, t: Mat, X_test: Mat | None = None, t_test: Mat | None = None, lr: float = 0.1, epochs: int = 2000) -> dict[str, list[float]]:
        """Entraîne le MLP par descente de gradient par lot.

        Si ``X_test``/``t_test`` sont fournis, le coût de test est aussi
        ajouté à l'historique.

        Parameters
        ----------
        X : Mat
            Batch d'entraînement, de forme (N, d).
        t : Mat
            Cibles d'entraînement, de forme (N, k).
        X_test : Mat, optionnel
            Batch de test, de forme (N_test, d).
        t_test : Mat, optionnel
            Cibles de test, de forme (N_test, k).
        lr : float, optionnel
            Pas d'apprentissage (défaut : 0.1).
        epochs : int, optionnel
            Nombre maximal d'epochs (défaut : 2000).

        Returns
        -------
        dict[str, list[float]]
            Historique des coûts ``{"train": [...], "test": [...]}``.
        """
        history = {"train": []}
        if X_test is not None:
            history["test"] = []
        for epoch in range(epochs):
            self.step(X, t, lr)
            history["train"].append(self.loss(X, t))
            if X_test is not None:
                history["test"].append(self.loss(X_test, t_test))
        return history


In [ ]:
class MLPMomentum(MLP):
    def __init__(self, sizes, activation="sigmoid", output_activation="linear", seed=0, momentum=0.9):
        super().__init__(sizes, activation, output_activation, seed)
        self.momentum = momentum
        self.vWs = [np.zeros_like(W) for W in self.Ws]
        self.vbs = [np.zeros_like(b) for b in self.bs]

    def step(self, X: Mat, t: Mat, lr: float) -> None:
        """Une étape de descente de gradient avec élan (momentum).

        v_l <- mu * v_l + lr * dW_l ; W_l <- W_l - v_l.
        """
        for l, (dW, db) in enumerate(self.backward(X, t)):
            self.vWs[l] = self.momentum * self.vWs[l] + lr * dW
            self.vbs[l] = self.momentum * self.vbs[l] + lr * db
            self.Ws[l] -= self.vWs[l]
            self.bs[l] -= self.vbs[l]


In [ ]:
def gen_2d(n=500, noise=0.1, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3.0, 3.0, size=(n, 2))
    t = (np.sin(X[:, 0]) * np.cos(X[:, 1])).reshape(-1, 1) + rng.normal(0.0, noise, size=(n, 1))
    return X, t

def plot_history(history, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    ax.plot(history["train"], label="train")
    if history["test"]:
        ax.plot(history["test"], label="test")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE")
    ax.set_yscale("log")
    ax.legend()
    ax.grid(True)
    return ax

def split(X: Mat, t: Mat, n_train: int, seed: int = 0) -> tuple[Mat, Mat, Mat, Mat]:
    """Sépare aléatoirement un jeu de données en entraînement et test.

    Les échantillons sont mélangés par une permutation déterministe
    (graine ``seed``), puis découpés : les ``n_train`` premiers forment
    l'ensemble d'entraînement, les suivants l'ensemble de test.

    Parameters
    ----------
    X : Mat
        Données d'entrée, de forme (N, d).
    t : Mat
        Cibles, de même forme que la sortie attendue (N, k).
    n_train : int
        Taille de l'ensemble d'entraînement.
    seed : int, optionnel
        Graine du générateur aléatoire (défaut : 0).

    Returns
    -------
    tuple[Mat, Mat, Mat, Mat]
        ``(X_train, t_train, X_test, t_test)``.
    """
    rng = np.random.default_rng(seed)
    perm = rng.permutation(X.shape[0])
    X_shuf, t_shuf = X[perm], t[perm]
    return X_shuf[:n_train], t_shuf[:n_train], X_shuf[n_train:], t_shuf[n_train:]


## Exercice 1 — Premier contact : avec/sans élan

### Énoncé

On reprend le jeu 2D standard du TD03 : cible $t = \sin(x_1)\cos(x_2)$, avec un bruit gaussien d'écart-type $\sigma = 0.1$ ; après découpage, **400** échantillons d'entraînement et **100** de test.

Architecture et conditions d'entraînement, **identiques** pour les deux variantes : réseau `[2, 16, 16, 1]`, tanh en caché, sortie **linéaire**, même graine, `lr = 0.2`, `epochs = 20000`.

> **Pronostic (a) — tempo train** — Avant tout entraînement : quel effet attendez-vous de $\mu = 0.9$ sur la vitesse de descente de la loss d'entraînement ?
>
> **Pronostic (b) — courbe test** — Et la courbe test : que fera chaque variante au fil des epochs (plateau ? remontée ?) ?
>
> *Notez vos pronostics : vous les confronterez aux courbes de la cellule suivante.*

### Corrigé

> **Exercice 1** — Générez le jeu 2D (bruit $\sigma = 0.1$), découpez-le avec `split` (400 en train, graine 1), puis entraînez la variante sans élan (`MLP`) et avec élan (`MLPMomentum`, $\mu = 0.9$). Tracez les deux historiques côte à côte (échelle log).


In [ ]:
# BEGIN
X4, t4 = gen_2d(noise=0.1)
X_train, t_train, X_test, t_test = split(X4, t4, n_train=400, seed=1)

lr, epochs = 0.2, 20000

model_plain = MLP([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0)
hist_plain = model_plain.fit_batch(X_train, t_train, X_test, t_test, lr=lr, epochs=epochs)

model_mom = MLPMomentum([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0, momentum=0.9)
hist_mom = model_mom.fit_batch(X_train, t_train, X_test, t_test, lr=lr, epochs=epochs)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
plot_history(hist_plain, axes[0])
plot_history(hist_mom, axes[1])
axes[0].set_title("sans élan (μ=0)")
axes[1].set_title("avec élan (μ=0.9)")
# END


### Lecture guidée du corrigé

Comparez avec vos pronostics. Les chiffres ci-dessous se lisent **après coup** sur la figure — l'énoncé ne les donnait pas à l'avance.

- **Train** : les deux courbes convergent de façon quasiment identique (valeurs finales ≈ 0.0050 sans élan, ≈ 0.0042 avec élan). Sur ce jeu, l'élan ne change presque rien à l'optimisation du train.
- **Test, sans élan** : la loss descend vers un plateau à ≈ 0.0068, atteint dans le dernier quart de l'entraînement, puis y reste. Aucune remontée : sur ce jeu riche (400 exemples) et bruité modérément, il n'y a tout simplement pas de surapprentissage.
- **Test, avec élan** : minimum ≈ 0.0078 atteint très tôt (vers l'epoch 3 549), puis remontée lente jusqu'à ≈ 0.0109 : les pas effectifs plus grands font que le réseau continue à progresser sur le train là où la version sans élan s'était stabilisée — le début d'un surapprentissage que l'élan accélère.

> **Phrase à retenir** — Sur ce jeu, l'élan ne dégrade pas grand-chose, mais il **fait bouger** la courbe test — les exercices suivants amplifient ce phénomène.


## Exercice 2 — La vallée mal conditionnée

### Énoncé

Le générateur `gen_vallee` (fourni dans le corrigé) produit $n = 200$ points :

- $x_1 \sim \mathcal{U}(-1,\,1)$ et $x_2 \sim \mathcal{U}(-80,\,80)$ : deux axes d'échelles très différentes ;
- cible $t = 0.5\,x_1 + 0.02\,x_2 + \varepsilon$, avec $\varepsilon \sim \mathcal{N}(0,\,0.05^2)$ ;
- **aucune normalisation des entrées**.

Le réseau est **linéaire** : `[2, 1]` (une seule couche, qui est la couche de sortie, activation de
sortie linéaire) — un poids par axe, pas de non-linéarité. Les deux variantes partent de la même
graine.

> **Exercice 2 (a) — pronostic** — Avant tout entraînement : avec `lr = 7e-4`, que feront les
> variantes **sans** puis **avec** élan ($\mu = 0.9$) ? Laquelle convergera, et à quel **tempo
> relatif** sur le train ?
>
> **Exercice 2 (b) — chronomètre** — « Plus vite » se mesure mieux par le **nombre d'epochs pour
> passer sous un seuil** de loss que par la valeur à l'epoch finale. Écrivez
> `epochs_to_threshold(hist, thr=0.005)` : le premier index de l'historique d'entraînement où la
> loss passe sous `thr` (`None` sinon).
>
> **Exercice 2 (c) — monter le pas** — Passez `lr = 1.3e-3` et recommencez les deux variantes :
> que fait la variante sans élan ? et celle avec élan ?
>
> **Exercice 2 (d) — option corrigée** — Visualisez la trajectoire des poids $(w_1, w_2)$ des deux
> variantes par-dessus le paysage de perte du réseau linéaire : que fait chaque variante dans la
> vallée ?
>
> **Exercice 2 (e) — discussion** — Que changerait la **standardisation des entrées** ?
> Formulez votre réponse avant de lire le test rapide en fin d'exercice.

### Corrigé — partie 1

> **Exercice 2** — Générez la vallée avec `gen_vallee()` (`n=200`, `scale=80.0`, `noise=0.05`,
> `seed=3`). Entraînez le réseau `[2, 1]` **sans élan** (`MLP`) puis **avec élan**
> (`MLPMomentum`, $\mu = 0.9$), même graine `seed=0`, `lr = 7e-4`, 6000 epochs ; chronométrez
> les deux runs avec `epochs_to_threshold` et tracez les historiques côte à côte (échelle log,
> seuil partagé à $0.005$).
>
> Puis recommencez à `lr = 1.3e-3` sur 2000 epochs : observez ce que fait chaque variante.


In [ ]:
# BEGIN
def gen_vallee(n=200, scale=80.0, noise=0.05, seed=3):
    """Vallée mal conditionnée : deux axes d'échelles très différentes.

    x1 ~ U(-1, 1), x2 ~ U(-scale, scale) ; t = 0.5·x1 + 0.02·x2 + ε ~ N(0, noise²).
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(-1.0, 1.0, size=n)
    x2 = rng.uniform(-scale, scale, size=n)
    t = 0.5 * x1 + 0.02 * x2 + rng.normal(0.0, noise, size=n)
    return np.column_stack([x1, x2]), t.reshape(-1, 1)


def epochs_to_threshold(hist, thr=0.005):
    """Premier index où la loss d'entraînement passe sous ``thr`` (None sinon)."""
    for i, loss_train in enumerate(hist["train"]):
        if loss_train < thr:
            return i
    return None


lr, epochs = 7e-4, 6000
X_v, t_v = gen_vallee()

model_plain = MLP([2, 1], output_activation="linear", seed=0)
hist_plain_v = model_plain.fit_batch(X_v, t_v, lr=lr, epochs=epochs)

model_mom = MLPMomentum([2, 1], output_activation="linear", seed=0, momentum=0.9)
hist_mom_v = model_mom.fit_batch(X_v, t_v, lr=lr, epochs=epochs)

ep_plain_v = epochs_to_threshold(hist_plain_v)
ep_mom_v = epochs_to_threshold(hist_mom_v)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, hist, ep, titre, annot in (
    (axes[0], hist_plain_v, ep_plain_v, "sans élan (μ=0)", (ep_plain_v - 150, 0.2, "right")),
    (axes[1], hist_mom_v, ep_mom_v, "avec élan (μ=0.9)", (ep_mom_v + 450, 0.2, "left")),
):
    ax.plot(hist["train"])
    ax.axhline(0.005, color="crimson", ls=":", lw=1)
    if ep is not None:
        ax.axvline(ep, color="crimson", ls="--", lw=1)
        ax.annotate(f"seuil 0.005 : epoch {ep}", xy=(ep, 0.005),
                    xytext=annot[:2], ha=annot[2], color="crimson",
                    arrowprops=dict(arrowstyle="->", color="crimson"))
    ax.set_yscale("log")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE (train)")
    ax.set_title(titre)
    ax.grid(True)
# END


In [ ]:
# BEGIN
lr_hi, epochs_hi = 1.3e-3, 2000

# le run sans élan va diverger (pertes non finies ou très grandes) : on coupe les avertissements
with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    hist_plain_hi = MLP([2, 1], output_activation="linear", seed=0).fit_batch(
        X_v, t_v, lr=lr_hi, epochs=epochs_hi)

hist_mom_hi = MLPMomentum([2, 1], output_activation="linear", seed=0, momentum=0.9).fit_batch(
    X_v, t_v, lr=lr_hi, epochs=epochs_hi)

# pertes sans élan : non finies ou > 50 -> np.nan (trous dans la ligne)
plain_hi = np.array([
    v if np.isfinite(v) and v <= 50.0 else np.nan
    for v in hist_plain_hi["train"]
])
a_diverge = bool(np.isnan(plain_hi).any())
ep_mom_hi = epochs_to_threshold(hist_mom_hi)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].plot(plain_hi, ".-", ms=3)  # nan = trous : la courbe sort du cadre
axes[0].set_xlim(0, epochs_hi)
axes[0].set_title("sans élan (μ=0) : divergence")
axes[1].plot(hist_mom_hi["train"])
axes[1].set_title("avec élan (μ=0.9) : converge")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE (train)")
    ax.grid(True)

if a_diverge:
    premier_div = int(np.argmax(np.isnan(plain_hi)))
    axes[0].axvline(premier_div, color="crimson", ls="--", lw=1)
    axes[0].text(premier_div + 40, 12.0,
                 f"perte > 50 dès l'epoch {premier_div}\n"
                 "(au-delà : hors du cadre, la loss explose)",
                 color="crimson", va="top")
if ep_mom_hi is not None:
    axes[1].axvline(ep_mom_hi, color="green", ls="--", lw=1)
    axes[1].text(ep_mom_hi + 60, 1.5,
                 f"seuil atteint : epoch {ep_mom_hi}",
                 color="green", va="top")
# END


### Réponses attendues

- **(a)** Le run **sans élan** finit par converger, mais il **zigzague systématiquement sur l'axe
  raide** $w_2$ (celui des grandes valeurs de $x_2$) tout en rampant le long de l'axe facile
  $w_1$ : il met **4 858 epochs** à passer sous $0.005$. La variante **avec élan** fait dix fois
  mieux (**485 epochs**) : la vitesse $v \leftarrow \mu v + \eta\,\nabla E$ **moyenne les
  gradients** — les composantes oscillantes de l'axe raide se compensent d'un coup sur l'autre,
  celles de l'axe facile s'accumulent — ce qui accélère le long de l'axe facile et amortit le
  zigzag.
- **(b)** `epochs_to_threshold` mesure le premier passage sous le seuil : sans élan, des
  **milliers** d'epochs (4 858 sur nos runs) ; avec élan, des **centaines** (485). Les
  annotations des figures affichent les valeurs mesurées sur les runs — l'énoncé ne les donnait
  pas à l'avance.
- **(c)** À `lr = 1.3e-3`, le **sans élan diverge** : le pas dépasse la limite de stabilité de
  l'axe raide, les pertes explosent vers `inf`/`nan` dès les premières epochs. L'**élan amortit**
  ce dépassement (bref dépassement visible sur $w_2$) et converge : seuil atteint vers
  l'**epoch 261**.
- **(d)** et **(e)** : les corrigés des deux options sont ci-dessous (trajectoire des poids, puis
  standardisation).


### Corrigé (option) — la trajectoire des poids

Paysage du modèle linéaire sans biais :

$$E(w) = \tfrac{1}{2}\,\mathrm{mean}\big((X w - t)^2\big)$$

tracé sur la grille $w_1 \in [-0.65,\,0.65] \times w_2 \in [-0.09,\,0.09]$, avec les
trajectoires des poids des deux variantes (`lr = 7e-4`, 2500 epochs, enregistrement des poids tous
les 20 epochs via une boucle manuelle sur `model.step(X_v, t_v, lr)`). Départ : le $w$ initial
aléatoire (graine 0) ; étoile verte : l'optimum théorique $(0.5,\ 0.02)$. Avec `momentum=0`,
`MLPMomentum` redonne exactement la descente sans élan.


In [ ]:
# BEGIN
def paysage_vallee(X, t, w1s, w2s):
    """Paysage du modèle linéaire sans biais : E(w) = 0.5 · mean((X·w − t)²)."""
    E = np.empty((w2s.size, w1s.size))
    for j, w1 in enumerate(w1s):
        for k, w2 in enumerate(w2s):
            E[k, j] = 0.5 * np.mean((w1 * X[:, 0] + w2 * X[:, 1] - t[:, 0]) ** 2)
    return E


def trajectoire_poids(momentum, lr=7e-4, epochs=2500, pas=20, seed=0):
    """Entraîne le réseau [2, 1] et enregistre (w1, w2) tous les ``pas`` epochs."""
    model = MLPMomentum([2, 1], output_activation="linear", seed=seed, momentum=momentum)
    traj = [model.Ws[0].ravel().copy()]  # départ : w initial aléatoire
    for epoch in range(epochs):
        model.step(X_v, t_v, lr)
        if (epoch + 1) % pas == 0:
            traj.append(model.Ws[0].ravel().copy())
    traj.append(model.Ws[0].ravel().copy())
    return np.array(traj)


w1s = np.linspace(-0.65, 0.65, 201)
w2s = np.linspace(-0.09, 0.09, 201)
E = paysage_vallee(X_v, t_v, w1s, w2s)

traj_plain_p = trajectoire_poids(0.0)
traj_mom_p = trajectoire_poids(0.9)
opt = (0.5, 0.02)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, traj, titre in (
    (axes[0], traj_plain_p, "sans élan (μ=0)"),
    (axes[1], traj_mom_p, "avec élan (μ=0.9)"),
):
    cf = ax.contourf(w1s, w2s, E, levels=40, cmap="Blues")
    ax.plot(traj[:, 0], traj[:, 1], ".-", color="crimson", lw=1, ms=3)
    ax.plot(traj[0, 0], traj[0, 1], "o", color="black", ms=6, label="départ (w initial)")
    ax.plot(*opt, "*", color="limegreen", ms=14, label="optimum (0.5, 0.02)")
    ax.set_xlabel("$w_1$ (poids de $x_1$)")
    ax.set_ylabel("$w_2$ (poids de $x_2$)")
    ax.set_title(titre)
    ax.legend(loc="upper left", fontsize="small")
fig.colorbar(cf, ax=axes, label="E(w)")
# END


### Pour aller plus loin (corrigé) — et la standardisation ?

Réflexion : standardiser les entrées ($x_j \leadsto (x_j - \bar{x}_j)\, /\, \sigma_j$) met les
deux axes à la même échelle. Le malconditionnement — le rapport des courbures de l'ordre de
6400 entre les deux axes — s'évanouit : on peut relever le `lr` sans divergence, la descente
avance à la même vitesse dans toutes les directions et **l'écart dramatique de tempo que l'élan
compensait disparaît** (l'élan redevient accessoire pour ce cas).

Vérifions-le : même réseau, même cible, entrées standardisées, `lr` relevé de deux ordres de
grandeur.


In [ ]:
# BEGIN
x1_std = (X_v[:, 0] - X_v[:, 0].mean()) / X_v[:, 0].std()
x2_std = (X_v[:, 1] - X_v[:, 1].mean()) / X_v[:, 1].std()
X_std = np.column_stack([x1_std, x2_std])

lr_std = 0.1  # impossible avant standardisation : l'axe raide diverge dès ~1e-3

hist_std_plain = MLP([2, 1], output_activation="linear", seed=0).fit_batch(
    X_std, t_v, lr=lr_std, epochs=6000)
hist_std_mom = MLPMomentum([2, 1], output_activation="linear", seed=0, momentum=0.9).fit_batch(
    X_std, t_v, lr=lr_std, epochs=6000)

print("standardisé, lr = 0.1 : seuil 0.005 atteint à l'epoch",
      epochs_to_threshold(hist_std_plain), "sans élan vs",
      epochs_to_threshold(hist_std_mom), "avec élan")
# END


## Exercice 3 — Le bruit mémorisé

### Énoncé

On revient à la fonction 2D du TD03, mais cette fois le jeu la rend **mémorisable** : le générateur
`gen_2d` est appelé avec un bruit **4 fois plus fort** (`noise = 0.4`, soit $\sigma = 0.4$ au lieu
de $0.1$), et le découpage n'alloue que **100 échantillons d'entraînement** (les 400 autres restent
en test). Peu d'exemples + beaucoup de bruit : le régime où un réseau assez gros peut apprendre le
bruit lui-même. Réseau `[2, 16, 16, 1]` (tanh, sortie linéaire), `lr = 0.05`, `epochs = 12000`,
même graine modèle 0 pour les deux variantes : sans élan, avec élan ($\mu = 0.9$).

> **Pronostic (a) — écart train/test** — Pour chaque variante : jusqu'où la courbe train va-t-elle
> descendre, et que fera la courbe test (plateau bas ? point bas puis remontée ?) ? Prédisez
> l'écart train/test au bout des 12 000 epochs, variante par variante.
>
> **Pronostic (b) — courbe de généralisation idéale** — Définissez en une phrase ce que la courbe
> test d'un entraînement **idéalement réglé** devrait faire **au point bas** de la courbe test
> (là où l'on choisirait de s'arrêter si on monitorait le test).
>
> *Notez vos pronostics : vous les confronterez aux courbes de la cellule suivante.*

### Corrigé

> **Exercice 3** — Générez le jeu très bruité avec `gen_2d(noise=0.4, seed=1)`, découpez-le avec
> `split(..., n_train=100, seed=2)` (100 en train, 400 en test), puis entraînez sans élan (`MLP`) et
> avec élan (`MLPMomentum`, $\mu = 0.9$) sur **exactement les mêmes données** (même graine modèle
> 0), avec `lr = 0.05` et `epochs = 12000`. Tracez les deux historiques côte à côte (échelle log).

In [ ]:
# BEGIN
Xn, tn = gen_2d(noise=0.4, seed=1)
Xn_train, tn_train, Xn_test, tn_test = split(Xn, tn, n_train=100, seed=2)
print("train :", Xn_train.shape, " test :", Xn_test.shape)

lr, epochs = 0.05, 12000

model_plain = MLP([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0)
hist_plain_n = model_plain.fit_batch(Xn_train, tn_train, Xn_test, tn_test, lr=lr, epochs=epochs)

model_mom = MLPMomentum([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0, momentum=0.9)
hist_mom_n = model_mom.fit_batch(Xn_train, tn_train, Xn_test, tn_test, lr=lr, epochs=epochs)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
plot_history(hist_plain_n, axes[0])
plot_history(hist_mom_n, axes[1])
axes[0].set_title("sans élan (μ=0)")
axes[1].set_title("avec élan (μ=0.9)")
# END

### Lecture guidée du corrigé

Comparez avec vos pronostics. Les chiffres ci-dessous se lisent **après coup** sur la figure —
l'énoncé ne les donnait pas à l'avance.

- **Repère utile** : avec $t = f(x) + \varepsilon$ et $\varepsilon \sim \mathcal{N}(0,\,0.4^2)$, la
  meilleure test loss atteignable (le **plancher du bruit**, pour un modèle qui connaît $f$ mais pas
  les $\varepsilon$) vaut $\tfrac{1}{2}\,\sigma^2 = 0.08$. Aucune courbe test ne s'installe
  durablement sous cette valeur.
- **Sans élan** : la loss train finit ≈ **0.0444** — **sous le plancher du bruit**. Une seule
  explication : le réseau mémorise le bruit de ses 100 exemples. La courbe test passe par son point
  bas ≈ **0.0947** vers l'**epoch 168** (juste au-dessus du plancher : un excellent point d'arrêt),
  puis remonte lentement jusqu'à ≈ **0.1546** en fin d'entraînement : surapprentissage, mais
  **doux**.
- **Avec élan** : la loss train plonge à ≈ **0.0024** (mémorisation quasi parfaite, bruit compris)
  et la courbe test culmine bien au-dessus : point bas ≈ **0.1099** dès l'**epoch 95**, puis
  remontée jusqu'à ≈ **1.2204** — 13× le point bas du sans-élan. Surapprentissage **brutal**.
- **Réponse au pronostic (a)** : sur ce cas, l'élan **accélère l'optimisation de tout, y compris
  l'apprentissage du bruit** — le train descend plus vite et plus bas, la test finit bien plus haut.
  L'élan aggrave le surapprentissage terminal **sans améliorer le point bas test** (0.1099 contre
  0.0947).
- **Réponse au pronostic (b)** : au point bas de la courbe test, l'entraînement idéalement réglé
  devrait **s'installer** : descendre jusqu'au meilleur compromis biais/variance — ici, juste
  au-dessus du plancher du bruit — puis **y tenir un plateau propre**, sans remonter ensuite.

> **Phrase à retenir** — Sur ce cas, l'élan n'est pas un remède au surapprentissage : en allant plus
> vite, il apprend aussi plus vite ce qu'il **ne faut pas** apprendre, le bruit propre à
> l'échantillon d'entraînement. (Un constat sur ce cas, pas une loi universelle.)

### Pour aller plus loin (corrigé) — la mini-challenge

Le point bas du sans-élan est `te_min` ≈ 0.0947. Avec de l'élan, mais **mieux réglé**, peut-on le battre ?

> **Mini-challenge (corrigé)** — Trouvez une configuration **avec élan** ($0 < \mu < 1$) dont la
> courbe test descende sous 0.0947 — ou expliquez pourquoi vous abandonnez. Gardez le même jeu, le
> même split, la même graine modèle 0 et `epochs = 12000` : ne changez que le couple ($\mu$, `lr`).

Le corrigé essaie trois candidats classiques : réduire le `lr` de moitié ($\mu = 0.9$, `lr = 0.02`),
le réduire encore (`lr = 0.01`), puis modérer l'élan lui-même ($\mu = 0.5$, `lr = 0.02`).

In [ ]:
# BEGIN
candidats = [(0.9, 0.02), (0.9, 0.01), (0.5, 0.02)]
hists_c = []
print(f"{'μ':>4} {'lr':>5} {'te_min':>8} {'epoch argmin':>12} {'te_fin':>8}")
for mu, lrc in candidats:
    model_c = MLPMomentum([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0, momentum=mu)
    hist_c = model_c.fit_batch(Xn_train, tn_train, Xn_test, tn_test, lr=lrc, epochs=epochs)
    hists_c.append(hist_c)
    te = hist_c["test"]
    i_min = int(np.argmin(te))
    print(f"{mu:>4} {lrc:>5} {te[i_min]:>8.4f} {i_min:>12} {te[-1]:>8.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, hist_c, (mu, lrc) in zip(axes, hists_c, candidats):
    plot_history(hist_c, ax)
    te = hist_c["test"]
    i_min = int(np.argmin(te))
    ax.set_title(f"μ={mu}, lr={lrc}")
    ax.annotate(f"te_min {te[i_min]:.4f} @ ep. {i_min}", xy=(i_min, te[i_min]),
                xytext=(i_min + 1500, te[i_min] * 3), color="crimson", fontsize="small",
                arrowprops=dict(arrowstyle="->", color="crimson"))
# END

**Bilan du challenge** — Aucun des trois essais ne passe sous 0.0947 : les points bas s'étagent
de **0.0949** ($\mu = 0.5$, `lr = 0.02`, epoch 215) à **0.1033** ($\mu = 0.9$, `lr = 0.02`,
epoch 126), et la test **finale** reste pire (0.14 à 0.51) dès que le `lr` avec élan est encore un
peu fort. Abandonner était la réponse : **sur ce cas**, aucun couple ($\mu$, `lr`) essayé ne bat le
sans-élan — c'est le résultat. L'élan sert à fitter le train, bruit compris, plus vite ; c'est
précisément ce qui se retourne contre la généralisation ici.

Passons à l'exercice 4.

## Exercice 4 (optionnel) — Mini-batchs : l'élan lisse-t-il vraiment ?

### Énoncé

Jusqu'ici, chaque `step` calculait le gradient sur le **train complet** : l'oscillation observée à
l'Exercice 2 (le zigzag de la vallée mal conditionnée) venait du **paysage**, pas de l'estimation.
Passez maintenant au **mini-batch** : le gradient n'est plus qu'une **estimation**, calculée sur un
petit bloc d'exemples tiré au hasard à chaque mise à jour — une composante **aléatoire** vient
s'ajouter au gradient. Le folklore « l'élan lisse les oscillations » survit-il quand
l'oscillation est un bruit, et plus un zigzag ?

> **Exercice 4 — à écrire et à exécuter** — Écrivez une fonction d'entraînement par mini-batchs avec
> la signature **exacte** :
>
> ```python
> def fit_mini(model, X, t, Xte, tte, batch_size, lr, epochs, seed=0):
> ```
>
> qui retourne, **dans cet ordre**, trois `np.ndarray` :
>
> - `per_batch` : la perte du **mini-batch courant**, enregistrée à chaque **mise à jour** ;
> - `train_ep` : la perte sur le **train complet**, enregistrée à la fin de chaque epoch ;
> - `test_ep` : la perte sur le jeu de test, à la même cadence.
>
> Détails imposés :
>
> - le générateur `rng = np.random.default_rng(seed)` est créé **une seule fois, au début de la
>   fonction** (hors de la boucle des epochs) : c'est lui qui rend reproductible la suite des
>   mélanges d'indices au fil de l'entraînement ;
> - à chaque epoch, **mélangez les indices** (`rng.permutation`), puis parcourez les **sous-blocs
>   consécutifs** du mélange — de taille `batch_size`, le dernier pouvant être plus petit — en
>   appliquant `model.step` sur chacun.
>
> **Puis** comparez $\mu = 0$ (`MLP`) et $\mu = 0.9$ (`MLPMomentum`), même graine modèle 0 :
> jeu `gen_2d(noise=0.1)`, découpage 100 train / 400 test (graine 2), réseau `[2, 16, 16, 1]`
> tanh / sortie linéaire, `batch_size = 16`, `lr = 0.1`, `epochs = 300`.
>
> Tracez : en rangée du haut, la perte **par mini-batch** de chaque run (axe $y$ borné à
> $[0,\,1]$ pour rester lisible devant les pointes de début d'entraînement) ; en rangée du bas,
> les courbes **par epoch** train/test de chaque run (échelle log).
>
> **Pronostic (a) — bruit par mini-batch** — Le folklore prédit-il que la courbe `per_batch` sera
> **moins** bruitée avec $\mu = 0.9$ ? Regardez surtout la fin de l'entraînement.
>
> **Pronostic (b) — courbe test** — Et la courbe test par epoch : plafonnera-t-elle à la même
> hauteur dans les deux variantes ?
>
> *Notez vos pronostics : vous les confronterez aux courbes de la cellule suivante.*

### Corrigé


In [ ]:
# BEGIN
def fit_mini(model, X, t, Xte, tte, batch_size, lr, epochs, seed=0):
    """Entraîne ``model`` par mini-batchs.

    À chaque epoch, les indices sont mélangés puis découpés en sous-blocs
    consécutifs de ``batch_size`` (le dernier peut être plus petit) ; une
    mise à jour ``model.step`` est effectuée sur chaque sous-bloc.

    Parameters
    ----------
    model : MLP
        Modèle à entraîner (``MLP`` ou ``MLPMomentum``).
    X : Mat
        Données d'entraînement, de forme (N, d).
    t : Mat
        Cibles d'entraînement (N, k).
    Xte : Mat
        Données de test (N_test, d).
    tte : Mat
        Cibles de test (N_test, k).
    batch_size : int
        Taille d'un mini-batch.
    lr : float
        Pas d'apprentissage.
    epochs : int
        Nombre d'epochs (chaque epoch voit le train complet, mélangé).
    seed : int, optionnel
        Graine du mélange des indices (défault : 0).

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray]
        ``(per_batch, train_ep, test_ep)`` : perte du bloc courant à chaque
        mise à jour, puis perte train complète et perte test à chaque epoch.
    """
    rng = np.random.default_rng(seed)  # créé UNE fois : la suite des mélanges est reproductible
    n = X.shape[0]
    n_batches = int(np.ceil(n / batch_size))
    per_batch, train_ep, test_ep = [], [], []
    for _ in range(epochs):
        perm = rng.permutation(n)
        for k in range(n_batches):
            idx = perm[k * batch_size:(k + 1) * batch_size]
            model.step(X[idx], t[idx], lr)
            per_batch.append(model.loss(X[idx], t[idx]))
        train_ep.append(model.loss(X, t))
        test_ep.append(model.loss(Xte, tte))
    return np.asarray(per_batch), np.asarray(train_ep), np.asarray(test_ep)


Xm, tm = gen_2d(noise=0.1)
Xm_train, tm_train, Xm_test, tm_test = split(Xm, tm, n_train=100, seed=2)
print("train :", Xm_train.shape, " test :", Xm_test.shape)

batch_size, lr, epochs = 16, 0.1, 300

model_mini_plain = MLP([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0)
pb_plain, tr_plain, te_plain = fit_mini(
    model_mini_plain, Xm_train, tm_train, Xm_test, tm_test, batch_size, lr, epochs, seed=0)

model_mini_mom = MLPMomentum([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0, momentum=0.9)
pb_mom, tr_mom, te_mom = fit_mini(
    model_mini_mom, Xm_train, tm_train, Xm_test, tm_test, batch_size, lr, epochs, seed=0)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

axes[0, 0].plot(pb_plain, lw=0.5)
axes[0, 0].set_ylim(0, 1.0)
axes[0, 0].set_title("perte par mini-batch — sans élan (μ=0)")

axes[0, 1].plot(pb_mom, lw=0.5)
axes[0, 1].set_ylim(0, 1.0)
axes[0, 1].set_title("perte par mini-batch — avec élan (μ=0.9)")

axes[1, 0].plot(tr_plain, label="train")
axes[1, 0].plot(te_plain, label="test")
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("par epoch — sans élan (μ=0)")
axes[1, 0].legend()

axes[1, 1].plot(tr_mom, label="train")
axes[1, 1].plot(te_mom, label="test")
axes[1, 1].set_yscale("log")
axes[1, 1].set_title("par epoch — avec élan (μ=0.9)")
axes[1, 1].legend()

for ax in axes[0, :]:
    ax.set_xlabel("mise à jour")
for ax in axes[1, :]:
    ax.set_xlabel("epoch")

print("std per_batch, 2000 dernières mises à jour : μ=0 : %.4f | μ=0.9 : %.4f"
      % (np.std(pb_plain[-2000:]), np.std(pb_mom[-2000:])))
print("test finale (par epoch) : μ=0 : %.4f | μ=0.9 : %.4f" % (te_plain[-1], te_mom[-1]))
# END


### Lecture du corrigé

Comparez avec vos pronostics. Les chiffres ci-dessous se lisent **après coup** sur la figure —
l'énoncé ne les donnait pas à l'avance.

- **Perte par mini-batch, sans élan ($\mu = 0$)** : à $\approx 2\,100$ mises à jour en tout
  (7 sous-blocs par epoch : 6 blocs de 16 + 1 bloc de 4), la courbe se densifie vite et le bruit
  **s'amortit** : sur les 2 000 dernières mises à jour, l'écart-type des pertes vaut ≈ **0.005**
  et les pointes restent sous ≈ 0.06.
- **Perte par mini-batch, avec élan ($\mu = 0.9$)** : la moyenne descend un peu plus bas, mais le
  bruit **grossit** : écart-type ≈ **0.017** — environ **3×** celui du sans-élan — et des pointes
  qui culminent au-delà de **0.15** sur la même fenêtre. **Contre-exemple au folklore** :
  l'élan n'a pas lissé l'oscillation stochastique, il l'a **amplifiée**.
- **Courbes par epoch** : côté train, peu de différence (finale ≈ **0.0098** sans élan contre
  ≈ **0.0088** avec : l'élan optimise toujours un peu mieux, comme à l'Exercice 2). Côté test, la
  courbe avec élan **plafonne plus haut** : test finale ≈ **0.0165** contre ≈ **0.0134**, et son
  point bas (≈ **0.0115** à l'epoch 210) ne descend pas sous celui du sans-élan
  (≈ **0.0110** à l'epoch 234).
- **Réponse au folklore : distinguez les deux sources d'oscillation.** L'élan **accumule**
  les gradients ; quand l'oscillation vient du **paysage** (zigzag **déterministe** entre deux
  axes d'échelles très différentes, Exercice 2), les gradients successifs pointent systématiquement
  dans la même direction et l'accumulateur glisse dans la vallée : le folklore est alors **vrai**.
  Mais quand l'oscillation vient de l'**estimation du gradient** (chaque mini-batch ne voit qu'un
  sous-échantillon), la composante aléatoire de chaque estimateur entre elle aussi dans la vitesse
  $v$, où elle **s'accumule** (chaque mise à jour d'hier y pèse encore $0.9^k$, au lieu d'être
  moyennée avec les suivantes) : le folklore est **faux** — à ce régime, l'élan rend les pertes
  par batch plus bruitées et la test finale plus haute.

> **Phrase à retenir** — « L'élan lisse les oscillations » : vrai pour le zigzag **systématique**
> hérité du paysage, faux pour le bruit **aléatoire** hérité du mini-batch — **sur ce régime**,
> il amplifie au contraire le bruit des pertes par batch et plafonne la test plus haut.


## Bilan de la section élan

Quatre exercices, quatre régimes, **la même mise à jour** $v \leftarrow \mu v + \eta\,\nabla E$.
Le tableau reprend les mesures des corrigés (mêmes chiffres que dans les « Lecture guidée » ci-dessus) :

| Exercice | Effet sur train | Effet sur test | Interprétation |
|---|---|---|---|
| **Ex. 1** — jeu 2D standard, riche (400 ex. bien vus) | Quasiment équivalent : ≈ 0.0050 sans élan contre ≈ 0.0042 avec — rien à gagner, rien à perdre | Point bas semblable, puis **remontée légère** avec élan : plateau ≈ 0.0068 (sans) contre te_min ≈ 0.0078 à l'epoch 3 549 → ≈ 0.0109 (avec) | L'élan ne dégrade rien ici, mais il **fait bouger** la courbe test là où le sans-élan s'était stabilisé : même sans surapprentissage visible, il continue de faire progresser le train |
| **Ex. 2** — vallée mal conditionnée | **Accéléré ×10** : 4 858 epochs → 485 pour passer sous 0.005 ; à `lr = 1.3e-3`, divergence du sans-élan là où l'élan converge (seuil à l'epoch 261) | Point bas test atteint plus tôt, au tempo du train | L'élan **moyenne les gradients** : l'axe facile s'accumule (accélération), le zigzag systématique de l'axe raide se compense (amortissement) |
| **Ex. 3** — bruit mémorisé, 100 ex. d'entraînement | Train fin ≈ 0.0444 → ≈ 0.0024 (**×18** de mieux — mémorisation quasi parfaite, bruit compris) | te_fin ≈ 0.1546 → ≈ 1.2204 (**×8** en fin d'entraînement) ; te_min **pas amélioré** (≈ 0.0947 vs ≈ 0.1099) | L'élan accélère l'optimisation de **tout**, y compris la mémorisation du bruit : sur ce cas, il **aggrave** le surapprentissage terminal sans améliorer le point bas test |
| **Ex. 4** — mini-batchs (bruit d'estimation) | Train par epoch peu différent (≈ 0.0098 vs ≈ 0.0088), mais **bruit par batch amplifié** : écart-type tardif ≈ 0.005 → ≈ 0.017 | Test plafonne **plus haut** : ≈ 0.0134 (sans) contre ≈ 0.0165 (avec) | Le folklore « l'élan lisse » est **vrai pour le zigzag déterministe** du paysage (Ex. 2), **faux pour le bruit aléatoire** du SGD : la composante aléatoire s'accumule dans $v$ au lieu d'être moyennée |

> **Question de synthèse (a)** — En une phrase chacun : ce que l'élan **garantit**, ce qu'il
> **ne garantit pas**.
>
> **Question de synthèse (b)** — À quelles conditions un couple de courbes train/test permet-il de
> conclure à un **effet de l'élan sur la généralisation** ? (Que change une graine unique ? un pas
> unique ? Faudrait-il plusieurs graines, plusieurs pas ?)


## Ouverture (bonus) — jusqu'où le dépassement reste-t-il amorti ?

> **Ouverture — la vallée pas à pas avec $\mu$** — Sur l'Exercice 2, remplacez
> $\mu = 0.9$ par $\mu \in \{0.3,\ 0.5,\ 0.99\}$ (mêmes données, même graine, `lr` inchangé).
> Pour $\mu = 0.99$ : le pas effectif $\eta/(1-\mu)$ devient 10× le `lr` — jusqu'où le
> **dépassement amplifié** sur l'axe raide reste-t-il amorti ? Tracez les trajectoires des poids
> (corrigé de l'option d) comme au corrigé : à partir de quel $\mu$ l'amortissement cesse-t-il ?

*(Question d'extension, sans corrigé ici — cf. Exercice 2 pour la méthode.)*
